# Ensemble learning III: voting, stacking, and champion–challenger evidence

**Lecture 15 · Notebook 02 · CMOR 438 / INDE 577**

Forests combine related randomized trees; boosting builds a sequential additive
model. This final laboratory asks when and how to combine **heterogeneous**
learners without leaking their training performance into a meta-model.

> **Can complementary models improve multiclass wine classification, and what
> evidence justifies promoting the ensemble over a simpler champion?**

## How to use this notebook

Run all cells in the **Rice DSM** kernel. Track which rows each fitted component
has seen. If you cannot state that boundary, do not trust the stack.

**Route:** diversity audit → hard and soft voting → out-of-fold feature
construction → stacking → calibration and multiclass metrics → champion–
challenger gate → project handoff.

## Learning objectives

You will be able to:

1. distinguish hard voting, soft voting, blending, and stacking;
2. construct out-of-fold (OOF) meta-features without training-row leakage;
3. explain why class order and calibration are API contracts;
4. evaluate multiclass ensembles beyond accuracy;
5. write a promotion gate including predictive and operational constraints; and
6. test a heterogeneous ensemble in the companion project.

## Why this matters

Different model families encode different inductive biases. A linear model may
extrapolate smoothly, nearest neighbors use local geometry, and trees partition
with rules. Their errors can be complementary—but combination can also preserve
shared bias, amplify leakage, and make ownership harder.

Wolpert's stacked generalization (1992) trains a second-level model on outputs
from first-level models. The decisive implementation detail is that every
training meta-feature must come from a model that did **not** train on that row.

**Professional practice.** Version the complete graph: preprocessing, base
models, class order, calibration, meta-model, threshold or decision rule, and
runtime dependencies. Test serialization and schema as one artifact.

In [ ]:
# Import the data-access and plotting tools shared by this laboratory.
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from rice_dsm.paths import course_database_path

RANDOM_STATE = 438
DATABASE_PATH = course_database_path()
assert DATABASE_PATH.is_file(), (
    "Course database missing. From the repository root run: "
    "uv run --locked python scripts/refresh_course.py"
)
sns.set_theme(style="whitegrid", context="notebook")
print("Database:", DATABASE_PATH)
print("Random seed:", RANDOM_STATE)

## 1. Worked example: chemical measurements of wine cultivars

The UCI Wine dataset contains 13 chemical measurements for wines from three
cultivars in one Italian region. It is real but small and historical. Our claim
is limited to a reproducible benchmark of multiclass methods—not authenticity,
quality, price, health, or modern geographic transport.

In [ ]:
# Query the course database rather than importing an opaque in-memory copy.
database_uri = f"file:{DATABASE_PATH.as_posix()}?mode=ro"
with sqlite3.connect(database_uri, uri=True) as connection:
    wine = pd.read_sql_query(
        "SELECT * FROM wine_observations ORDER BY observation_id", connection
    )

target_name = "cultivar_code"
target_label = "cultivar_label"
feature_names = [
    column for column in wine.columns
    if column not in {"observation_id", target_name, target_label}
]
display(wine.head())
display(wine[feature_names].describe().T)
print(wine[target_name].value_counts().sort_index())

assert wine.shape[0] == 178
assert len(feature_names) == 13
assert wine[feature_names + [target_name]].isna().sum().sum() == 0
assert wine[target_name].nunique() == 3

In [ ]:
# Look for scale differences and overlapping class structure.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.boxplot(data=wine, x=target_label, y="alcohol", ax=axes[0])
sns.scatterplot(data=wine, x="flavanoids", y="color_intensity",
                hue=target_label, palette="deep", ax=axes[1])
axes[0].set_title("Alcohol varies within and between cultivars")
axes[1].set_title("Two features reveal overlap and local structure")
plt.tight_layout(); plt.show()
assert wine["alcohol"].std() > 0

In [ ]:
from sklearn.model_selection import train_test_split

X = wine[feature_names].to_numpy()
y = wine[target_name].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.25, stratify=y, random_state=RANDOM_STATE
)
assert set(np.unique(y_train)) == set(np.unique(y_test)) == set(np.unique(y))

## 2. Establish diverse, competent base learners

Diversity alone is useless if members are poor. We pair models with appropriate
preprocessing: scaling belongs inside pipelines for logistic regression and KNN;
trees generally do not need it. Cross-validation predictions let us examine
competence and disagreement within the training boundary.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict

base_models = {
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
    "neighbors": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=9)),
    "forest": RandomForestClassifier(
        n_estimators=150, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
    ),
}
# Cross-validated labels reveal agreement without using final test outcomes.
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_labels = {}
# Every row is predicted by a fold model that did not train on it.
for name, model in base_models.items():
    oof_labels[name] = cross_val_predict(model, X_train, y_train, cv=folds, n_jobs=-1)

agreement = pd.DataFrame(oof_labels).corr(method=lambda a, b: np.mean(a == b))
display(agreement)
assert agreement.shape == (3, 3)
assert np.allclose(np.diag(agreement), 1.0)

## 3. Hard and soft voting

**Hard voting** takes the modal predicted label; it discards confidence.
**Soft voting** averages class probabilities and takes the largest mean. Soft
voting requires the same class order and probabilities with comparable meaning.
An overconfident, poorly calibrated member can dominate.

Weights are hyperparameters. If they are tuned after inspecting test results,
the test set has become training data.

In [ ]:
# Fit hard and soft votes under the same frozen train/test boundary.
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, log_loss

hard_vote = VotingClassifier(list(base_models.items()), voting="hard")
soft_vote = VotingClassifier(list(base_models.items()), voting="soft")
vote_rows = []
for name, model in {**base_models, "hard vote": hard_vote, "soft vote": soft_vote}.items():
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)
    row = {"model": name, "accuracy": accuracy_score(y_test, prediction)}
    if hasattr(model, "predict_proba"):
        row["log_loss"] = log_loss(y_test, model.predict_proba(X_test), labels=np.unique(y))
    vote_rows.append(row)
vote_results = pd.DataFrame(vote_rows).set_index("model")
display(vote_results.round(3))
assert vote_results.shape[0] == 5

## 4. Stacking without leakage

Naive procedure: fit each base model on all training rows, predict those same
rows, and train a meta-model on the predictions. This is leakage because
in-sample base predictions are systematically too optimistic.

Correct OOF procedure for $K$ folds:

1. reserve fold $k$;
2. fit every base learner on the other $K-1$ folds;
3. predict probabilities for fold $k$;
4. place those probabilities in the corresponding OOF rows;
5. fit the meta-model on the completed OOF matrix;
6. refit each base learner on all training rows for future test or production
   predictions.

With $C$ classes and $B$ base learners, the raw meta-matrix has $B\times C$
columns. Class order is part of the contract.

In [ ]:
from sklearn.base import clone

classes = np.unique(y_train)
oof_blocks = []
test_blocks = []
for name, model in base_models.items():
    # cross_val_predict ensures each training probability comes from a model
    # that did not fit the corresponding row.
    oof_probability = cross_val_predict(
        model, X_train, y_train, cv=folds, method="predict_proba", n_jobs=-1
    )
    fitted = clone(model).fit(X_train, y_train)
    assert np.array_equal(fitted.classes_, classes)
    oof_blocks.append(oof_probability)
    test_blocks.append(fitted.predict_proba(X_test))

X_meta_train = np.column_stack(oof_blocks)
X_meta_test = np.column_stack(test_blocks)
print("OOF meta-features:", X_meta_train.shape)
print("test meta-features:", X_meta_test.shape)
assert X_meta_train.shape == (len(X_train), len(base_models) * len(classes))
assert X_meta_test.shape[1] == X_meta_train.shape[1]

In [ ]:
# A regularized linear meta-model learns a restrained combination.
meta_model = LogisticRegression(max_iter=3000, C=0.5).fit(X_meta_train, y_train)
stack_probability = meta_model.predict_proba(X_meta_test)
stack_prediction = meta_model.predict(X_meta_test)
stack_row = pd.DataFrame([{
    "model": "manual OOF stack",
    "accuracy": accuracy_score(y_test, stack_prediction),
    "log_loss": log_loss(y_test, stack_probability, labels=classes),
}]).set_index("model")
display(pd.concat([vote_results, stack_row]).round(3))

assert stack_probability.shape == (len(X_test), len(classes))
np.testing.assert_allclose(stack_probability.sum(axis=1), 1.0)

## 5. Multiclass evidence and error context

Accuracy hides which cultivars are confused. Macro metrics weight classes
equally; weighted metrics follow observed support. Log loss evaluates the full
probability vector and strongly penalizes confident mistakes. Report the
confusion matrix and per-class precision, recall, and support alongside the
aggregate.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

print(classification_report(y_test, stack_prediction, digits=3))
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, stack_prediction, cmap="Blues", ax=ax)
ax.set_title("OOF stack: cultivar error context")
plt.tight_layout(); plt.show()
assert stack_prediction.shape == y_test.shape

## 6. Champion–challenger promotion gate

The champion is the approved model currently satisfying the product contract.
A challenger is evaluated under the same split, metrics, preprocessing boundary,
and resource budget. Promotion needs a practically meaningful improvement—not
merely the largest point estimate among many experiments.

A gate may require:

- noninferior macro recall and no unacceptable class regression;
- improved or noninferior log loss after calibration;
- bounded bootstrap uncertainty on the change;
- fit time, batch throughput, tail latency, memory, and artifact size limits;
- deterministic rebuild and successful serialization/schema tests;
- explanation of added ownership, dependency, and monitoring burden.

In [ ]:
# Encode a small, reviewable gate rather than promoting by intuition.
champion_accuracy = vote_results.loc["logistic", "accuracy"]
champion_log_loss = vote_results.loc["logistic", "log_loss"]
challenger_accuracy = stack_row.loc["manual OOF stack", "accuracy"]
challenger_log_loss = stack_row.loc["manual OOF stack", "log_loss"]

gate = {
    "accuracy_noninferior": challenger_accuracy >= champion_accuracy - 0.01,
    "log_loss_improves": challenger_log_loss < champion_log_loss,
    "probabilities_valid": np.allclose(stack_probability.sum(axis=1), 1.0),
}
display(pd.Series(gate, name="passed"))
print("promote:", all(gate.values()))
assert set(gate) == {"accuracy_noninferior", "log_loss_improves", "probabilities_valid"}

## Common failure modes and debugging

- **Meta-training performance is perfect:** confirm every meta-feature is OOF.
- **Probability columns silently swap:** persist and test class order for every
  component.
- **Scaled features leak folds:** put preprocessing inside each base pipeline.
- **The stack is worse:** complementary errors are not guaranteed; a complex
  combiner can overfit a small meta-dataset.
- **Serialized graph fails:** test the whole preprocessing–base–meta artifact in
  the supported runtime.
- **No one can own it:** operational complexity is part of model quality.

## Guided practice

Replace one base learner with a shallow gradient booster. Rebuild OOF features
from scratch, visualize pairwise disagreement, and evaluate whether diversity,
log loss, or only complexity changed. Then compare a constrained meta-model with
simple soft voting.

In [ ]:
# PRACTICE: verify that a proposed meta-matrix has no impossible values.
assert np.isfinite(X_meta_train).all()
assert ((0 <= X_meta_train) & (X_meta_train <= 1)).all()
for block in np.hsplit(X_meta_train, len(base_models)):
    np.testing.assert_allclose(block.sum(axis=1), 1.0)

## Independent exercise: student contribution test plan

In the companion project, add a heterogeneous ensemble only if the research
question justifies it. Put combination code under
`src/dsm_project/algorithms/student/ensembles.py` and tests under
`tests/test_student_ensembles.py`. Test OOF coverage exactly once per row,
meta-feature shape, class alignment, probability sums, reproducibility,
prediction-before-fit, serialization, and a deliberately leaky construction
that your test suite rejects. The applied notebook imports the package and
reports both predictive gain and added operational cost.

In [ ]:
# Companion-repository workflow (run in its terminal):
# git switch main
# git pull --ff-only
# git switch -c student/stacking-<short-question>
# uv sync --locked
# uv run ruff check .
# uv run pytest -q
# git add src tests notebooks data/fixtures
# git commit -m "Add leakage-safe ensemble for <question>"
# git push -u origin student/stacking-<short-question>
assert True

## Extension

Implement nested cross-validation for both base and meta-level choices, or study
time-aware and group-aware OOF construction. Explain why ordinary shuffled folds
would answer the wrong deployment question.

## Retrieval practice

1. Why can hard and soft voting disagree?
2. Why must training meta-features be out of fold?
3. What does one probability block contribute to the meta-matrix?
4. Why does scaling belong inside each cross-validation fit?
5. What evidence besides predictive quality belongs in a promotion gate?

## Takeaway

Heterogeneous ensembles help only when member competence and complementary error
survive a leakage-safe evaluation. Soft voting requires calibrated, aligned
probabilities; stacking requires OOF training features. Promotion is a systems
decision balancing predictive evidence against complexity and cost.

## Further reading

- Wolpert, “Stacked Generalization” (1992).
- scikit-learn [stacking documentation](https://scikit-learn.org/stable/modules/ensemble.html#stacked-generalization).
- scikit-learn [model evaluation guide](https://scikit-learn.org/stable/modules/model_evaluation.html).
- Python [`pickle`](https://docs.python.org/3/library/pickle.html) security warning;
  treat model artifact loading as a trust boundary.